In the space available below, include your variable statistics and the Visualizations associated with them. These should show information that will be pertinent to your analysis. Do not choose random categories to analyze and visualize. Below the visualizations, describe what you see in the data and how it helps you understand your question.

Requirements:

- 1 pearson r correlation coefficient
- 1 t-test
- 1 ANOVA
- 1 visualization for each of the different tests.
- There should be at least 3 visualizations, and at least 3 different types of visualizations (e.g. scatter plot, bar chart, box plot, histogram, etc.)

In [ ]:
# Is there a correlation between high ratings and show popularity?

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import ttest_ind, f_oneway

df = pd.read_csv('hulu.csv')

print("most common genres:")
genreCounts = df['show/genre'].value_counts().head(5)
print(genreCounts)

# pearson correlation, ep count vs ratings
plt.figure(figsize=(10, 6))
sns.scatterplot(x='show/episodes_count', y='show/rating', data=df)
plt.title('correlation episode count vs rating')
plt.xlabel('episodes ount')
plt.ylabel('rating count')

r, p = stats.pearsonr(df['show/episodes_count'].dropna(), df['show/rating'].dropna())
plt.show()
print(f"pearson: r = {r: .2f}, p = {p: .2f}")

# t test, comparing ep counts of two popular genres like reality and drama
df['isReality'] = df['show/genre'].str.contains('Reality and Game Shows', na=False)
df['isDrama'] = df['show/genre'].str.contains('Drama', na=False) & ~df['isReality']
reality = df[df['isReality']]['show/episodes_count'].dropna()
drama = df[df['isDrama']]['show/episodes_count'].dropna()

plt.figure(figsize=(8, 5))
means = [reality.mean(), drama.mean()]
groups = ['Reality Shows', 'Other Shows']
sns.barplot(x=groups, y=means)
plt.title('episode count, reality vs drama')
plt.ylabel('avg episode count')

t_stat, t_p = ttest_ind(reality, drama, equal_var=False)
plt.show()
print(f"t-test: t={t_stat:.4f}, p-value={t_p:.4f}")

# anova, compare ratings across genres
ratingGroups = ['isReality', 'isDrama']
groups = [df[df[group]]['show/rating'].dropna() for group in ratingGroups]

plt.figure(figsize=(12, 6))
sns.boxplot(x='isReality', y='show/rating', data=df)
plt.title('Rating Distribution by Content Type')
plt.xlabel('Is Reality Show')
plt.ylabel('Rating')
plt.xticks([0, 1], ['False', 'True'])

f, p = stats.f_oneway(*groups)
alpha = 0.05
dfb = len(ratingGroups) - 1
dfw = df['show/rating'].dropna().count() - len(ratingGroups)
crit_f = stats.f.ppf(1 - alpha, dfb, dfw)
plt.show()
print(f"f= {f: .2f} p= {p: .2f} crit_f= {crit_f: .2f}")

print("\nSummary of Statistical Tests:")
print(f"1. pearson Correlation between episode count and rating: r={r:.2f}, p={p:.2f}")
print(f"2. t-test comparing episode counts in Reality vs Drama shows: t={t_stat:.2f}, p={t_p:.2f}")
print(f"3. anova comparing ratings between Reality and other shows: f={f:.2f}, p={p:.2f}, crit_f={crit_f:.2f}")

# Q: Is there a correlation between high ratings and show popularity?

For the pearson correlation (r = 0.02, p = 0.50), it seems theres no correlation between episode count and rating. The p-value also shows this lack of relationship is not by chance. So show popularity measured by episode count does not predict rating quality.

For the t-test comparison of reality vs drama shows (t = -2.30, p = 0.02), the negative t-value means that drama shows tend to have more episodes than reality shows. And the p-value suggests this difference is "statistically significant", so this only shows that different genres can have different production patterns. 

For the ANOVA testing of comparing ratings by genre (F = 441.25, p < 0.001, crit_f = 3.85), the p-value is very small, indicating that the ratings are significantly different across the genres. Also, the critical F-value is much smaller than the calculated F-value, which means we can reject the null hypothesis that all groups have the same mean rating. This ultimately means that it depends on the type of genre when it comes to how strong the ratings are.

Overall, the data suggests that high ratings and show popularity (episode count) are not directly correlated. Instead, genre appears to be a more important factor in both how shows are produced (episode count) and how they're rated. Different genre expectations most likely influence viewer ratings more than a show's popularity.